Ce code permet de créer le panel final qui nous servira pour les estimations.
À partir des fichiers panel_va_STAN.csv & panel_cit1.csv, on procède à plusieurs modifications :
 - création des secteurs agrégés C & H
 - création du vecteur d'intensité de R&D
 - création des agrégats de R&D : intra-sectoriel, extra-sectoriel, domestique et étranger
 - ajout d'une DUMMY pour la crise de 2009
 - calcul de l'indicateur HHI (Herfindahl-Hirschmann) par secteur
 - séparation en 2 panels distincts selon l'intensité de R&D moyenne pour chaque pays, pour chaque base de données

Ce code retourne des panels longs pour chaque base prêt à être utilisé pour les estimations.

Version 2.0 : lit le panel du run choisi dans `project_settings.py` (RD_UNIT, DIMENSION, TECH), lit l'intensité `intensite_propre.csv` (RDV / PRODV, `prépa_va_STAN`) et écrit les panels dans `runs/<run>/panels/`.

In [67]:
import numpy as np
import pandas as pd
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, METRIC_WANTED, METRIC_SPE, YEARS, RESULTS_DIR, RUN_DIR, PANEL_CODE_DIR, CHOC, CHOC_SCENARIO
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO})" if CHOC else "") + f" -> {RESULTS_DIR}")
import os

CHOC = False -> D:/ENTICE/rendu_fin_stage/choc_tfp/runs/SRDQ (c,s)/


In [68]:
df = pd.read_csv(RESULTS_DIR + f"panel_estimation/panel_estimation_{BASE}.csv")   # written by calc_blocs2.0 for the run chosen in project_settings

In [69]:
print(df.columns)

Index(['Region', 'Sector', 'year', 'rd_same_sector_same_country_rd_dir_cit',
       'rd_same_sector_diff_country_rd_dir_cit',
       'rd_diff_sector_same_country_rd_dir_cit',
       'rd_diff_sector_diff_country_rd_dir_cit'],
      dtype='str')


In [70]:
df = df.rename(columns={"year": "annee"})

Création des secteurs agrégés

In [71]:
c_subsectors = df.loc[df['Sector'].str.startswith('C', na=False), 'Sector'].unique().tolist()

mask_c = df['Sector'].isin(c_subsectors)

phys = ["FIGARO", "EXIOBASE", "GTAP", "OECD_ICIO", "CAPITAL"]

if BASE in phys:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',

            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',

            'exports',
            'imports_final_demand',
            'imports_intermediate',
            'production']
else:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}']

c_agg = (
    df[mask_c]
    .groupby(['Region', 'annee'], as_index=False)
    [col]
    .sum()
)
c_agg['Sector'] = 'C'

df_final = pd.concat([df, c_agg], ignore_index=True)

In [72]:
if BASE in phys:
    h_subsectors = df.loc[df['Sector'].str.startswith('H', na=False), 'Sector'].unique().tolist()
    mask_h = df_final['Sector'].isin(h_subsectors)

    h_agg = (
        df_final[mask_h]
        .groupby(['Region', 'annee'], as_index=False)
        [col]
        .sum()
    )
    h_agg['Sector'] = 'H'

    df_final1 = pd.concat([df_final, h_agg], ignore_index=True)
else:
    df_final1 = df_final

In [73]:
os.makedirs(RESULTS_DIR + 'panels', exist_ok=True)
df_final1.to_csv(RESULTS_DIR + f'panels/panel_{BASE}_intermediaire_F.csv')

In [74]:
df_final1

,Region,Sector,annee,rd_same_sector_same_country_rd_dir_cit,rd_same_sector_diff_country_rd_dir_cit,rd_diff_sector_same_country_rd_dir_cit,rd_diff_sector_diff_country_rd_dir_cit
0,AT,C10_C12,1995,3.960341,2391.247974,23.855201,5068.364520
1,AT,C13_C15,1995,6.818776,581.465399,41.024306,8538.319391
2,AT,C16_C18,1995,9.997983,644.723868,47.798228,9204.283472
3,AT,C19,1995,0.328810,360.909959,23.153914,12757.197209
4,AT,C20+C21,1995,145.656872,28806.766210,33.151215,8295.454001
...,...,...,...,...,...,...,...
22451,US,C,2018,544560.379784,125274.671871,640664.820303,148713.184705
22452,US,C,2019,571013.023127,133229.156377,672011.257593,157296.855996
22453,US,C,2020,598112.812991,139341.263841,706675.412641,164842.533219
22454,US,C,2021,624016.540738,146013.756078,734004.853402,173178.588881


Découpe des différents panels

In [75]:
if BASE in phys:
    va_STAN = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/Panels_intermediaires/panel_va_STAN.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]

    df_final1 = df_final1.rename(columns={"production": f"production_{BASE}"})
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])
else:
    va_STAN = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/Panels_intermediaires/panel_cit1.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])

df

,Region,Sector,annee,VA_V,VA_Q,VA_costFactors,comp_emp,emp,net_stock_capital,production,...,w_l,c_u,produit,TFP_2,Y_L,K_L,rd_same_sector_same_country_rd_dir_cit,rd_same_sector_diff_country_rd_dir_cit,rd_diff_sector_same_country_rd_dir_cit,rd_diff_sector_diff_country_rd_dir_cit
0,AT,C,1995,3.254808e+04,3.733416e+04,3.183844e+04,20367.100000,1081.953922,7.911849e+04,8.140574e+04,...,18.824369,0.871804,5392.979431,6.922734,34.506240,73.125560,844.082040,162510.481910,631.210940,146469.327561
1,AT,C,1996,3.306172e+04,3.767389e+04,3.238062e+04,20181.200000,1065.267316,8.070817e+04,8.303116e+04,...,18.944728,0.877576,5750.123847,6.551840,35.365671,75.763306,889.174383,168901.714715,666.288361,151935.067586
2,AT,C,1997,3.438026e+04,3.867043e+04,3.361202e+04,20293.600000,1055.960426,8.239480e+04,8.681144e+04,...,19.218144,0.889058,6294.511431,6.143516,36.621099,78.028304,938.674465,180025.730237,704.546661,161230.214555
3,AT,C,1998,3.562284e+04,3.984189e+04,3.485630e+04,21158.200000,1059.099172,8.386637e+04,9.021730e+04,...,19.977544,0.894105,6250.043800,6.374657,37.618657,79.186512,1000.441694,188839.625295,751.947793,168624.822955
4,AT,C,1999,3.738677e+04,4.180839e+04,3.665022e+04,21638.200000,1053.325413,8.495945e+04,9.348271e+04,...,20.542749,0.894241,6694.338516,6.245335,39.691806,80.658315,1076.430006,197863.801497,810.193276,175620.638878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6355,US,H53+J,2015,1.079185e+06,9.873062e+05,1.064836e+06,551863.037997,11465.000000,1.997737e+06,1.800984e+06,...,48.134587,1.093061,142716.484414,6.917955,86.114799,174.246559,52389.986502,749.561456,122232.314194,8721.204899
6356,US,H53+J,2016,1.152394e+06,1.072515e+06,1.137069e+06,572171.420067,11801.000000,2.100906e+06,1.917937e+06,...,48.484994,1.074478,160333.571908,6.689273,90.883402,178.027836,61011.195780,829.399394,128772.133895,9199.907329
6357,US,H53+J,2017,1.199139e+06,1.150476e+06,1.182808e+06,607522.367361,12007.000000,2.215458e+06,2.000688e+06,...,50.597349,1.042298,157550.694617,7.302261,95.817133,184.513853,69615.881122,938.460749,133887.008216,9663.547285
6358,US,H53+J,2018,1.282444e+06,1.233441e+06,1.265176e+06,654032.653651,12071.000000,2.337561e+06,2.139995e+06,...,54.182143,1.039729,159370.315910,7.739464,102.182158,193.650979,81507.681133,993.866027,138211.894332,10197.841887


In [76]:
# embod
if BASE in phys:
    col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_F'] = df[col_F].sum(axis=1)
    df['rd_embod_D'] = df[col_D].sum(axis=1)

    col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_I'] = df[col_I].sum(axis=1)
    df['rd_embod_X'] = df[col_X].sum(axis=1)

    col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod'] = df[col_S].sum(axis=1)

#dir
col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_F'] = df[col_F].sum(axis=1)
df['rd_dir_D'] = df[col_D].sum(axis=1)

col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_I'] = df[col_I].sum(axis=1)
df['rd_dir_X'] = df[col_X].sum(axis=1)

col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir'] = df[col_S].sum(axis=1)

df["2009_crisis"]=(df["annee"]==2009).astype(int)

# col = 'production_EXIOBASE_mapF'
# df['share'] = df['production'] / df.groupby(['Sector', 'annee'])[col].transform('sum')
df['share'] = df['production'] / df.groupby(['Sector', 'annee'])['production'].transform('sum')
df['HHI'] = df.groupby(['Sector', 'annee'])['share'].transform(lambda s: (s**2).sum())

In [77]:
df = df.drop(columns=['share'])

On ajoute l'intensité en R&D

In [78]:
df['intensité'] = df[f'rd_{METRIC_WANTED}'] / df[f'production']

In [79]:
df.to_csv(RESULTS_DIR + f'panels/panel_{BASE}_F.csv', index=False)

Division des panels par base en deux groupes de pays en fonction de leur intensité de R&D moyenne

In [80]:
# R&D propre / production : fichier RDV / PRODV calculé une seule fois par prépa_va_STAN (indépendant de la base et de la technologie)
# rd_propre = flux de R&D en M€ courants, production = production STAN aux prix courants en M€ (taux de change de l'année), C et H inclus
df_int = (pd.read_csv(PANEL_CODE_DIR + "Panels_intermediaires/intensite_propre.csv")
            .rename(columns={"rd_propre_V": "rd_propre", "production_V": "production"}))

# vraie intensité = R&D propre du secteur / production du secteur
df_int["intensité_vraie"] = df_int["rd_propre"] / df_int["production"]

# intensité annuelle propre R/Y (en points de %), pour le terme croisé de l'estimation
df_int.loc[df_int["production"] <= 1.0, "intensité_vraie"] = np.nan   # production quasi nulle => intensité absurde
int_annual = (df_int[["Region", "Sector", "annee", "intensité_vraie"]]
              .rename(columns={"intensité_vraie": "int_propre"}))
int_annual["int_propre"] = 100 * int_annual["int_propre"]


groupe,faible,forte
Sector,,
A,11.0,12.0
B,11.0,12.0
C,11.0,12.0
C10_C12,11.0,12.0
C13_C15,11.0,12.0
C16_C18,11.0,12.0
C19,11.0,12.0
C20+C21,11.0,12.0
C22,11.0,12.0


In [81]:
# Panel complet, avec int_propre
df = df.merge(int_annual, on=["Region", "Sector", "annee"], how="left")

os.makedirs(RESULTS_DIR + "panels", exist_ok=True)

df.to_csv(RESULTS_DIR + f"panels/panel_{BASE}_F.csv", index=False)
